# 05 — Patient Similarity Network Analysis

This notebook builds and analyses the patient comorbidity risk network:

1. Feature vector construction from clinical metadata
2. Patient similarity graph construction (cosine similarity)
3. Louvain community detection
4. Centrality analysis (degree, betweenness, PageRank)
5. DR grade homophily measurement
6. High-risk cluster identification
7. Gephi export for interactive exploration

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from src.network.similarity import build_patient_network
from src.network.community import detect_communities, compute_centrality_metrics
from src.network.analysis import NetworkAnalyzer
from src.utils.config import load_config

cfg = load_config('../configs/default.yaml')
clinical_df = pd.read_csv('../data/metadata/clinical_metadata.csv')
print(f'Loaded {len(clinical_df)} patients')
clinical_df.head()

## 1. Feature Vector Construction

In [ ]:
feature_cols = [c for c in cfg['network']['clinical_features']
                if c in clinical_df.columns]
print(f'Using {len(feature_cols)} features: {feature_cols}')
clinical_df[feature_cols].describe().round(2)

## 2. Build Patient Similarity Graph

In [ ]:
dr_grades = clinical_df['dr_grade'].values if 'dr_grade' in clinical_df.columns else None

G = build_patient_network(
    clinical_df=clinical_df,
    feature_columns=feature_cols,
    threshold=cfg['network']['edge_threshold'],
    dr_grades=dr_grades,
)

print(f'\nGraph statistics:')
print(f'  Nodes: {G.number_of_nodes()}')
print(f'  Edges: {G.number_of_edges()}')
print(f'  Density: {nx.density(G):.4f}')
print(f'  Avg clustering: {nx.average_clustering(G):.4f}')

## 3. Community Detection (Louvain)

In [ ]:
partition, modularity = detect_communities(G)
n_communities = len(set(partition.values()))
print(f'Communities detected: {n_communities}')
print(f'Modularity score (Q): {modularity:.4f}')
print('  Q > 0.3 indicates meaningful community structure')

# Community size distribution
from collections import Counter
comm_sizes = Counter(partition.values())
print(f'\nCommunity sizes: {sorted(comm_sizes.values(), reverse=True)[:10]}')

## 4. Centrality Analysis

In [ ]:
centrality = compute_centrality_metrics(G)

# Top 10 most connected patients
top_degree = sorted(centrality['degree'].items(), key=lambda x: -x[1])[:10]
print('Top 10 by Degree Centrality:')
for pid, dc in top_degree:
    grade = G.nodes[pid].get('dr_grade', '?')
    print(f'  {pid}: degree={dc:.4f}  DR_grade={grade}')

# Top 10 bridge patients
top_between = sorted(centrality['betweenness'].items(), key=lambda x: -x[1])[:10]
print('\nTop 10 Bridge Patients (Betweenness Centrality):')
for pid, bc in top_between:
    grade = G.nodes[pid].get('dr_grade', '?')
    print(f'  {pid}: betweenness={bc:.4f}  DR_grade={grade}')

## 5. DR Grade Homophily

In [ ]:
analyzer = NetworkAnalyzer(
    clinical_df=clinical_df,
    feature_columns=feature_cols,
    threshold=cfg['network']['edge_threshold'],
    dr_grades=dr_grades,
)
homophily = analyzer.compute_dr_homophily()
print(f'DR Grade Homophily: {homophily:.4f}')
print('  1.0 = all edges connect same-grade patients')
print('  0.2 = random (expected if no structure)')

## 6. Network Visualizations

In [ ]:
Path('../outputs/network').mkdir(parents=True, exist_ok=True)

# Community view
analyzer.visualize_network(
    save_path='../outputs/network/patient_network_community.png',
    color_by='community', size_by='degree', layout='spring',
)

# DR grade view
analyzer.visualize_network(
    save_path='../outputs/network/patient_network_drgrade.png',
    color_by='dr_grade', size_by='betweenness', layout='spring',
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, fname, title in zip(axes,
    ['../outputs/network/patient_network_community.png',
     '../outputs/network/patient_network_drgrade.png'],
    ['Coloured by Community', 'Coloured by DR Grade']):
    import cv2
    img = cv2.cvtColor(cv2.imread(fname), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. High-Risk Cluster Identification

In [ ]:
high_risk = analyzer.get_high_risk_clusters()
print('High-risk communities (sorted by mean DR grade):\n')
for comm_id, stats in list(high_risk.items())[:8]:
    print(f'  Community {comm_id}:')
    print(f'    Size: {stats["size"]} patients')
    print(f'    Mean DR grade: {stats["mean_dr_grade"]:.2f}')
    print(f'    Severe fraction: {stats["severe_fraction"]:.1%}')

In [ ]:
# Gephi export
analyzer.export_gexf('../outputs/network/patient_graph.gexf')
print('GEXF exported — open in Gephi for interactive exploration')
print('Tip: In Gephi: Layout > ForceAtlas2, Appearance > Nodes > Colour by community')

## Summary

| Metric | Value | Interpretation |
|---|---|---|
| Modularity Q | *computed above* | > 0.3 = meaningful clusters |
| DR Homophily | *computed above* | > 0.5 = grade-based clustering |
| High-risk clusters | *computed above* | Communities with mean grade ≥ 3 |